# Itchy — Unpatch Layer Ablation

The unpatch layer (patch representation -> 12 byte predictions) is the bottleneck.
Test better decode strategies.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

Expected time: ~30 min

In [ ]:
!pip install -q torch numpy huggingface-hub sentencepiece
import torch, math, time, os, shutil, numpy as np
import torch.nn.functional as F
from torch import nn
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
print(f'GPU: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda')

In [ ]:
# Data
os.makedirs('data/bytes', exist_ok=True)
def dl(fn,sub,dst):
    if Path(dst).exists(): return
    c=str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn,subfolder=sub,repo_type='dataset')).resolve())
    try: os.link(c,dst)
    except: shutil.copy2(c,dst)
dl('fineweb_train_000000.bin','datasets/datasets/fineweb10B_sp1024','data/train.bin')
dl('fineweb_val_000000.bin','datasets/datasets/fineweb10B_sp1024','data/val.bin')
dl('fineweb_1024_bpe.model','datasets/tokenizers','data/tok.model')
def load_shard(p):
    h=np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4)
def write_shard(p,d):
    h=np.zeros(256,dtype='<i4');h[0]=20240520;h[1]=1;h[2]=len(d)
    with open(p,'wb') as f: f.write(h.tobytes());f.write(d.astype('<u2').tobytes())
if not Path('data/bytes/train.bin').exists():
    sp=spm.SentencePieceProcessor(model_file='data/tok.model')
    for s,d in [('data/train.bin','data/bytes/train.bin'),('data/val.bin','data/bytes/val.bin')]:
        print(f'Converting {s}...')
        tok=load_shard(s)
        bs=[np.frombuffer(sp.decode(tok[i:i+100000].tolist()).encode('utf-8'),dtype=np.uint8)
            for i in range(0,len(tok),100000)]
        write_shard(d,np.concatenate(bs))
def load_bytes(p):
    h=np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4).astype(np.int64)
train_data=load_bytes('data/bytes/train.bin')
val_data=load_bytes('data/bytes/val.bin')
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# ============================================================
# SHARED BUILDING BLOCKS
# ============================================================

PATCH = 12
DIM = 256
LAYERS = 8
NH, NKV = 8, 4
MLP_MULT = 3

class Rotary(nn.Module):
    def __init__(self,dim):
        super().__init__()
        inv_freq=1.0/(10000.0**(torch.arange(0,dim,2,dtype=torch.float32)/dim))
        self.register_buffer('inv_freq',inv_freq,persistent=False)
        self._c=None
    def forward(self,S,dev,dt):
        if self._c is None or self._c[0]!=S:
            t=torch.arange(S,device=dev,dtype=self.inv_freq.dtype)
            f=torch.outer(t,self.inv_freq.to(dev))
            self._c=(S,f.cos()[None,None,:,:],f.sin()[None,None,:,:])
        return self._c[1].to(dt),self._c[2].to(dt)

def rope(x,cos,sin):
    h=x.size(-1)//2
    x1,x2=x[...,:h],x[...,h:]
    return torch.cat((x1*cos+x2*sin,x1*(-sin)+x2*cos),dim=-1)

class Attn(nn.Module):
    def __init__(self,dim,nh,nkv):
        super().__init__()
        self.nh,self.nkv,self.hd=nh,nkv,dim//nh
        kv=nkv*self.hd
        self.cq=nn.Linear(dim,dim,bias=False)
        self.ck=nn.Linear(dim,kv,bias=False)
        self.cv=nn.Linear(dim,kv,bias=False)
        self.proj=nn.Linear(dim,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg=nn.Parameter(torch.full((nh,),1.5))
        self.rot=Rotary(self.hd)
    def forward(self,x):
        B,S,D=x.shape
        q=self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k=self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v=self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q,k=F.rms_norm(q,(self.hd,)),F.rms_norm(k,(self.hd,))
        c,s=self.rot(S,x.device,q.dtype)
        q,k=rope(q,c,s),rope(k,c,s)
        q=q*self.qg.to(q.dtype)[None,:,None,None]
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True,enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self,dim,mult):
        super().__init__()
        self.fc=nn.Linear(dim,dim*mult,bias=False)
        self.proj=nn.Linear(dim*mult,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self,x):
        return self.proj(F.leaky_relu(self.fc(x),0.5).square())

class Block(nn.Module):
    def __init__(self,dim,nh,nkv,mult):
        super().__init__()
        self.attn=Attn(dim,nh,nkv)
        self.mlp=MLP(dim,mult)
        self.as_=nn.Parameter(torch.ones(dim))
        self.ms=nn.Parameter(torch.ones(dim))
        self.rm=nn.Parameter(torch.stack((torch.ones(dim),torch.zeros(dim))))
    def forward(self,x,x0):
        m=self.rm.to(x.dtype)
        x=m[0][None,None,:]*x+m[1][None,None,:]*x0
        x=x+self.as_.to(x.dtype)[None,None,:]*self.attn(F.rms_norm(x,(x.size(-1),)))
        x=x+self.ms.to(x.dtype)[None,None,:]*self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x

def make_backbone(dim, layers, nh, nkv, mult, patch_size):
    """Shared backbone: byte embed + patch proj + transformer blocks."""
    return nn.ModuleDict({
        'be': nn.Embedding(260, dim),
        'pp': nn.Linear(dim * patch_size, dim, bias=False),
        'blocks': nn.ModuleList([Block(dim,nh,nkv,mult) for _ in range(layers)]),
        'sw': nn.ParameterList([nn.Parameter(torch.ones(dim)) for _ in range(layers//2)]),
    })

def run_backbone(bb, ids, dim, patch_size):
    B,S=ids.shape; P=patch_size
    x=bb['be'](ids).reshape(B,S//P,P*dim)
    x=F.rms_norm(bb['pp'](x),(dim,))
    x0=x; ne=len(bb['blocks'])//2; nd=len(bb['blocks'])-ne; skips=[]
    for i in range(ne): x=bb['blocks'][i](x,x0); skips.append(x)
    for i in range(nd):
        if skips: x=x+bb['sw'][i].to(x.dtype)[None,None,:]*skips.pop()
        x=bb['blocks'][ne+i](x,x0)
    return F.rms_norm(x,(x.size(-1),))

print('Blocks defined')

In [ ]:
# ============================================================
# UNPATCH VARIANTS
# ============================================================

# A. BASELINE: flat linear projection (current)
class Model_FlatLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = make_backbone(DIM, LAYERS, NH, NKV, MLP_MULT, PATCH)
        self.up = nn.Linear(DIM, PATCH*260, bias=False)
        self.sc = 30.0
    def forward(self, ids, tgt):
        x = run_backbone(self.bb, ids, DIM, PATCH)
        lo = self.up(x).reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

# B. POSITION EMBEDDINGS: add learned byte-position-within-patch before projection
class Model_PosEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = make_backbone(DIM, LAYERS, NH, NKV, MLP_MULT, PATCH)
        self.pos = nn.Parameter(torch.randn(PATCH, DIM) * 0.02)  # learned position per byte in patch
        self.head = nn.Linear(DIM, 260, bias=False)  # shared across positions
        self.sc = 30.0
    def forward(self, ids, tgt):
        x = run_backbone(self.bb, ids, DIM, PATCH)  # (B, n_patches, dim)
        B, NP, D = x.shape
        # Expand to per-byte: (B, n_patches, 12, dim)
        x = x.unsqueeze(2).expand(B, NP, PATCH, D) + self.pos[None, None, :, :]
        # Project each byte position to logits
        lo = self.head(x)  # (B, NP, 12, 260)
        lo = lo.reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

# C. POS EMBED + PER-POSITION MLP HEAD: each byte position gets a small MLP
class Model_PosHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = make_backbone(DIM, LAYERS, NH, NKV, MLP_MULT, PATCH)
        self.pos = nn.Parameter(torch.randn(PATCH, DIM) * 0.02)
        # Small MLP per position: dim -> 128 -> 260
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(DIM, 128, bias=False), nn.ReLU(), nn.Linear(128, 260, bias=False))
            for _ in range(PATCH)
        ])
        self.sc = 30.0
    def forward(self, ids, tgt):
        x = run_backbone(self.bb, ids, DIM, PATCH)  # (B, NP, dim)
        B, NP, D = x.shape
        logits = []
        for p in range(PATCH):
            xp = x + self.pos[p][None, None, :]  # (B, NP, dim)
            logits.append(self.heads[p](xp))  # (B, NP, 260)
        lo = torch.stack(logits, dim=2)  # (B, NP, 12, 260)
        lo = lo.reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

# D. AUTOREGRESSIVE MINI-DECODER: tiny transformer that predicts bytes sequentially within patch
class MiniDecoder(nn.Module):
    def __init__(self, dim, out_dim=260):
        super().__init__()
        self.dim = dim
        self.byte_embed = nn.Embedding(260, dim)  # for autoregressive input
        self.pos = nn.Parameter(torch.randn(PATCH, dim) * 0.02)
        # 1-layer causal self-attention
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.mlp_fc = nn.Linear(dim, dim*2, bias=False)
        self.mlp_proj = nn.Linear(dim*2, dim, bias=False)
        self.head = nn.Linear(dim, out_dim, bias=False)
        self.nh = max(1, dim // 32)
        self.hd = dim // self.nh

    def forward(self, patch_repr, target_bytes=None):
        """patch_repr: (B*NP, dim). target_bytes: (B*NP, 12) for teacher forcing."""
        BNP = patch_repr.shape[0]
        # Build input sequence: [patch_repr, byte0_embed, byte1_embed, ..., byte10_embed]
        # Predict: [byte0, byte1, ..., byte11]
        # Teacher forcing: use target bytes shifted right
        if target_bytes is not None:
            # Shift right: input is [0, tgt[0], tgt[1], ..., tgt[10]]
            shifted = torch.cat([torch.zeros(BNP, 1, dtype=torch.long, device=target_bytes.device),
                                 target_bytes[:, :-1]], dim=1)  # (BNP, 12)
            byte_emb = self.byte_embed(shifted)  # (BNP, 12, dim)
        else:
            byte_emb = torch.zeros(BNP, PATCH, self.dim, device=patch_repr.device, dtype=patch_repr.dtype)
        # Add patch representation and position embeddings
        x = byte_emb + patch_repr.unsqueeze(1) + self.pos[None, :, :]  # (BNP, 12, dim)
        # 1-layer causal self-attention
        S = PATCH
        q = self.q(F.rms_norm(x, (self.dim,))).reshape(BNP, S, self.nh, self.hd).transpose(1,2)
        k = self.k(F.rms_norm(x, (self.dim,))).reshape(BNP, S, self.nh, self.hd).transpose(1,2)
        v = self.v(F.rms_norm(x, (self.dim,))).reshape(BNP, S, self.nh, self.hd).transpose(1,2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1,2).reshape(BNP, S, self.dim))
        x = x + self.mlp_proj(F.relu(self.mlp_fc(F.rms_norm(x, (self.dim,)))))
        return self.head(x)  # (BNP, 12, 260)

class Model_AutoDecode(nn.Module):
    def __init__(self, dec_dim=128):
        super().__init__()
        self.bb = make_backbone(DIM, LAYERS, NH, NKV, MLP_MULT, PATCH)
        self.dec_proj = nn.Linear(DIM, dec_dim, bias=False)  # project patch repr to decoder dim
        self.decoder = MiniDecoder(dec_dim)
        self.sc = 30.0
    def forward(self, ids, tgt):
        x = run_backbone(self.bb, ids, DIM, PATCH)  # (B, NP, dim)
        B, NP, D = x.shape
        pr = self.dec_proj(x).reshape(B*NP, -1)  # (B*NP, dec_dim)
        tgt_bytes = tgt.reshape(B, NP, PATCH).reshape(B*NP, PATCH)  # (B*NP, 12)
        lo = self.decoder(pr, tgt_bytes)  # (B*NP, 12, 260)
        lo = lo.reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

# E. MIXTURE OF EXPERTS UNPATCH: 3 specialist decoders + router
class Model_MoEUnpatch(nn.Module):
    def __init__(self, n_experts=3):
        super().__init__()
        self.bb = make_backbone(DIM, LAYERS, NH, NKV, MLP_MULT, PATCH)
        self.n_experts = n_experts
        self.pos = nn.Parameter(torch.randn(PATCH, DIM) * 0.02)
        # Each expert: dim -> 260
        self.experts = nn.ModuleList([nn.Linear(DIM, 260, bias=False) for _ in range(n_experts)])
        # Router: dim -> n_experts (per patch)
        self.router = nn.Linear(DIM, n_experts, bias=False)
        self.sc = 30.0
    def forward(self, ids, tgt):
        x = run_backbone(self.bb, ids, DIM, PATCH)  # (B, NP, dim)
        B, NP, D = x.shape
        # Router weights per patch
        weights = F.softmax(self.router(x), dim=-1)  # (B, NP, n_experts)
        # Expand to per-byte
        x_expanded = x.unsqueeze(2).expand(B, NP, PATCH, D) + self.pos[None, None, :, :]
        # Expert predictions
        expert_logits = torch.stack([e(x_expanded) for e in self.experts], dim=-1)  # (B,NP,12,260,n_exp)
        # Weight and sum experts
        w = weights[:, :, None, None, :]  # (B, NP, 1, 1, n_experts)
        lo = (expert_logits * w).sum(dim=-1)  # (B, NP, 12, 260)
        lo = lo.reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('All models defined')

In [ ]:
# ============================================================
# HARNESS
# ============================================================

SEQ_LEN = (504 // PATCH) * PATCH  # 504, divisible by 12
BATCH = 8192
STEPS = 3000
LR = 3e-4

def train_and_val(name, model):
    model = model.to(device).bfloat16()
    p = n_params(model)
    print(f'\n{"="*60}')
    print(f'{name}: {p:,} params ({p/1e6:.1f}M)')
    print(f'{"="*60}')
    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.01)
    pos=0;t0=time.time();losses=[]
    for step in range(1,STEPS+1):
        usable=(BATCH//SEQ_LEN)*SEQ_LEN
        end=pos+usable+1
        if end>len(train_data): pos=0;end=usable+1
        chunk=train_data[pos:end];pos=end-1
        x=torch.tensor(chunk[:-1].reshape(-1,SEQ_LEN),device=device)
        y=torch.tensor(chunk[1:].reshape(-1,SEQ_LEN),device=device)
        opt.zero_grad()
        with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
            loss=model(x,y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step%1000==0:
            avg=np.mean(losses[-1000:])
            print(f'  step {step} | bpb {avg/math.log(2):.4f} | {time.time()-t0:.0f}s')
    tt=time.time()-t0
    vl=[]
    for i in range(0,min(200*SEQ_LEN,len(val_data)-SEQ_LEN-1),SEQ_LEN):
        chunk=val_data[i:i+SEQ_LEN+1]
        x=torch.tensor(chunk[:-1].reshape(1,-1),device=device)
        y=torch.tensor(chunk[1:].reshape(1,-1),device=device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
                vl.append(model(x,y).item())
    val=np.mean(vl);bpb=val/math.log(2)
    print(f'  >>> VAL BPB: {bpb:.4f} | {p/1e6:.1f}M | {tt:.0f}s')
    return {'name':name,'params':p,'val_bpb':bpb,'time':tt}

print('Harness ready')

In [ ]:
# ============================================================
# RUN
# ============================================================

results = []

# A. Baseline: flat linear unpatch
torch.manual_seed(42)
results.append(train_and_val('A. Flat linear (current)', Model_FlatLinear()))

# B. + Position embeddings within patch
torch.manual_seed(42)
results.append(train_and_val('B. +Pos embed + shared head', Model_PosEmbed()))

# C. + Position + per-position MLP heads
torch.manual_seed(42)
results.append(train_and_val('C. +Pos + per-pos MLP heads', Model_PosHead()))

# D. Autoregressive mini-decoder (128d)
torch.manual_seed(42)
results.append(train_and_val('D. Auto-regressive decoder', Model_AutoDecode(dec_dim=128)))

# E. Mixture of Experts unpatch (3 experts)
torch.manual_seed(42)
results.append(train_and_val('E. MoE unpatch (3 experts)', Model_MoEUnpatch(n_experts=3)))

In [ ]:
# ============================================================
# RESULTS
# ============================================================

print()
print('='*70)
print('UNPATCH LAYER ABLATION RESULTS')
print('='*70)
print(f'{"Config":<32} {"Params":>8} {"Val BPB":>9} {"Delta":>9} {"Time":>6}')
print('-'*70)

ref=results[0]['val_bpb']
best=min(r['val_bpb'] for r in results)
for r in results:
    d=r['val_bpb']-ref
    tag=''
    if d==0: tag=' <-- current'
    elif r['val_bpb']==best: tag=' *** BEST'
    elif d<-0.001: tag=' +'
    elif d>0.001: tag=' -'
    print(f'{r["name"]:<32} {r["params"]/1e6:>7.1f}M {r["val_bpb"]:>9.4f} {d:>+9.4f} {r["time"]:>5.0f}s{tag}')

print()
b=min(results,key=lambda r:r['val_bpb'])
print(f'BEST: {b["name"]} at {b["val_bpb"]:.4f} BPB')
print(f'Improvement over flat linear: {ref-b["val_bpb"]:+.4f} BPB')